# 1. 数据加载

导入所有依赖库，读取Kaggle比赛数据集。

In [1]:
from __future__ import annotations
from pathlib import Path
import numpy as np
import pandas as pd

from catboost import CatBoostClassifier, Pool
from sklearn.metrics import accuracy_score
from sklearn.model_selection import StratifiedKFold

In [ ]:

TRAIN_PATH = Path(r"E:\MLW\demo\train.csv")
TEST_PATH  = Path(r"E:\MLW\demo\test.csv")

# 读取原始数据
train_df = pd.read_csv(TRAIN_PATH)
test_df  = pd.read_csv(TEST_PATH)
print(f"训练集: {train_df.shape}, 测试集: {test_df.shape}")

训练集: (8693, 14), 测试集: (4277, 13)


# 2. 数据预处理
进行预处理，处理缺失值，软规则补全CryoSleep。

In [3]:
# 五项消费列名（后续多处复用）
SPEND_COLS = ["RoomService", "FoodCourt", "ShoppingMall", "Spa", "VRDeck"]


def _split_or_missing(series, sep, n_parts):
    """拆分结构型字符串列；原值缺失时用占位符填充。"""
    placeholder = sep.join(["Missing"] * n_parts)
    return series.fillna(placeholder).astype(str).str.split(sep, expand=True)


def add_structural_features(df):
    """
    从PassengerId、Cabin、Name中提取结构特征：
    PassengerId -> GroupID / GroupPos
    Cabin       -> Deck / CabinNum / Side
    Name        -> LastName
    """
    out = df.copy()
    pid_parts = _split_or_missing(out["PassengerId"], "_", 2)
    out["GroupID"]  = pid_parts[0].astype(str)
    out["GroupPos"] = pd.to_numeric(pid_parts[1], errors="coerce")

    cabin_parts = _split_or_missing(out["Cabin"], "/", 3)
    out["Deck"]     = cabin_parts[0].astype(str)
    out["CabinNum"] = pd.to_numeric(cabin_parts[1], errors="coerce")
    out["Side"]     = cabin_parts[2].astype(str)

    out["Name"]     = out["Name"].fillna("Missing Missing").astype(str)
    out["LastName"] = out["Name"].str.split().str[-1].astype(str)
    out["Age"] = pd.to_numeric(out["Age"], errors="coerce")
    for col in SPEND_COLS:
        out[col] = pd.to_numeric(out[col], errors="coerce")
    return out


def apply_soft_cryo_imputation(df):
    """
    软规则补全CryoSleep：
    CryoSleep缺失 且 消费总额>0 => 补False
    （有消费记录的人不可能处于冷冻睡眠状态）
    """
    out = df.copy()
    raw_spend_sum = out[SPEND_COLS].fillna(0).sum(axis=1)
    mask = out["CryoSleep"].isna() & (raw_spend_sum > 0)
    out.loc[mask, "CryoSleep"] = False
    # 消费列NaN统一补0（冷冻状态或未消费）
    for col in SPEND_COLS:
        out[col] = out[col].fillna(0)
    return out

# 3. 特征工程

构造聚合特征（TotalSpend、NoSpend、AgeBin），并对剩余缺失值做兜底填充。

In [ ]:
# 年龄分箱边界（CatBoost主线V1.2标准配置）
AGE_BIN_EDGES  = [-1.0, 12.0, 18.0, 25.0, 40.0, 60.0, 200.0]
AGE_BIN_LABELS = ["child", "teen", "young", "adult", "mid", "senior"]


def add_aggregate_features(df):
    """
    TotalSpend：五项消费之和（反映消费能力和冷冻状态）
    NoSpend   ：是否零消费（与CryoSleep强相关的二值特征）
    AgeBin    ：年龄分箱（减少连续年龄的过拟合风险）
    """
    out = df.copy()
    out["TotalSpend"] = out[SPEND_COLS].sum(axis=1)
    out["NoSpend"]    = (out["TotalSpend"] == 0).astype(int)
    out["Age"]        = out["Age"].fillna(out["Age"].median())
    out["AgeBin"] = pd.cut(
        out["Age"], bins=AGE_BIN_EDGES, labels=AGE_BIN_LABELS, include_lowest=True
    ).astype("object")
    return out


def final_fill(df):
    """兜底填补：数值列用中位数，类别列用字符串Missing。"""
    out = df.copy()
    for col in ["Age", "GroupPos", "CabinNum", "TotalSpend", "NoSpend"]:
        out[col] = pd.to_numeric(out[col], errors="coerce").fillna(out[col].median())
    for col in ["HomePlanet", "CryoSleep", "Destination", "VIP",
                "GroupID", "Deck", "Side", "LastName", "AgeBin"]:
        out[col] = out[col].fillna("Missing").astype(str)
    return out


def prepare_train_test(train_df, test_df):
    """
    做特征工程：
    保证两边的类别值域、分箱边界、缺失填充口径完全一致。
    """
    train_tmp = train_df.copy(); train_tmp["_is_train"] = 1
    test_tmp  = test_df.copy();  test_tmp["_is_train"]  = 0
    test_tmp["Transported"] = float("nan")

    full = pd.concat([train_tmp, test_tmp], axis=0, ignore_index=True)
    full = add_structural_features(full)
    full = apply_soft_cryo_imputation(full)
    full = add_aggregate_features(full)
    full = final_fill(full)

    train_p = full[full["_is_train"] == 1].copy().drop(columns=["_is_train"])
    test_p  = full[full["_is_train"] == 0].copy().drop(columns=["_is_train"])

    # 19个核心特征（CatBoost主线V1.2标准配置）
    feature_cols = [
        "HomePlanet", "CryoSleep", "Destination", "Age", "VIP",
        "RoomService", "FoodCourt", "ShoppingMall", "Spa", "VRDeck",
        "GroupID", "GroupPos", "Deck", "CabinNum", "Side",
        "LastName", "TotalSpend", "NoSpend", "AgeBin",
    ]
    # CatBoost通过Pool的cat_features参数原生支持字符串类别，无需编码
    categorical_cols = [
        "HomePlanet", "CryoSleep", "Destination", "VIP",
        "GroupID", "Deck", "Side", "LastName", "AgeBin",
    ]
    return train_p, test_p, feature_cols, categorical_cols


# 执行特征工程
train_p, test_p, feature_cols, categorical_cols = prepare_train_test(train_df, test_df)
print(f"特征数: {len(feature_cols)}, 类别特征数: {len(categorical_cols)}")

特征数: 19, 类别特征数: 9


# 4. 模型配置

CatBoost超参数（主线V1.2基线参数）。

In [5]:
# CatBoost超参数（V1.2主线基线参数）
CAT_PARAMS = dict(
    iterations          = 300,    # 最大树的数量
    depth               = 6,      # 树的最大深度（控制模型复杂度）
    learning_rate       = 0.05,   # 学习率（较小值+更多树=更稳定）
    l2_leaf_reg         = 3.0,    # L2正则化系数（防止过拟合）
    loss_function       = "Logloss",
    eval_metric         = "Accuracy",
    random_seed         = 42,     # 固定随机种子，保证可复现
    verbose             = False,
)

# 交叉验证设置
N_SPLITS              = 5    # 5折交叉验证
RANDOM_STATE          = 42
EARLY_STOPPING_ROUNDS = 100  # 验证集100轮不提升则停止

print("模型: CatBoost V1.2 主线基线")
print(f"CV策略: {N_SPLITS}折 StratifiedKFold")

模型: CatBoost V1.2 主线基线
CV策略: 5折 StratifiedKFold


# 5. 训练

定义模型构建函数。CatBoost通过Pool原生支持字符串类别特征，无需手动编码。

In [6]:
def build_catboost_model():
    """构建CatBoost分类器实例（每折重新初始化）。"""
    return CatBoostClassifier(**CAT_PARAMS)

# 6. 交叉验证

5折StratifiedKFold：每折独立训练，用early stopping防过拟合，汇总OOF预测评估泛化能力。
CatBoost通过Pool + cat_features索引直接接受字符串类别列，无需额外编码。

In [7]:
def cross_validate_and_predict(train_p, test_p, feature_cols, categorical_cols):
    """
    5折分层交叉验证：
    - StratifiedKFold：保证每折正负比例与整体一致
    - early_stopping_rounds：验证集Accuracy 100轮不提升则停止，避免过拟合
    - OOF(Out-of-Fold)：每条训练样本只在没见过它的折上预测，
      汇总后的OOF准确率是模型泛化能力的无偏估计
    - CatBoost Pool：传入cat_features索引，直接处理字符串类别列
    """
    X      = train_p[feature_cols].copy()
    y      = train_p["Transported"].astype(int).copy()
    X_test = test_p[feature_cols].copy()

    # 类别列在feature_cols中的整数索引（Pool要求传索引而非列名）
    cat_feature_indices = [feature_cols.index(c) for c in categorical_cols]

    oof_proba        = np.zeros(len(train_p), dtype=float)
    test_proba_folds = []

    cv = StratifiedKFold(n_splits=N_SPLITS, shuffle=True, random_state=RANDOM_STATE)

    for fold, (tr_idx, va_idx) in enumerate(cv.split(X, y), start=1):
        X_tr, y_tr = X.iloc[tr_idx], y.iloc[tr_idx]
        X_va, y_va = X.iloc[va_idx],  y.iloc[va_idx]

        train_pool = Pool(X_tr, y_tr, cat_features=cat_feature_indices)
        valid_pool = Pool(X_va, y_va, cat_features=cat_feature_indices)
        test_pool  = Pool(X_test,     cat_features=cat_feature_indices)

        model = build_catboost_model()
        model.fit(
            train_pool,
            eval_set=valid_pool,
            use_best_model=True,
            early_stopping_rounds=EARLY_STOPPING_ROUNDS,
        )

        va_proba  = model.predict_proba(valid_pool)[:, 1]
        te_proba  = model.predict_proba(test_pool)[:, 1]

        fold_acc = accuracy_score(y_va, (va_proba >= 0.5).astype(int))
        oof_proba[va_idx] = va_proba
        test_proba_folds.append(te_proba)
        print(f"Fold {fold}: accuracy={fold_acc:.5f}, best_iter={model.get_best_iteration()}")

    # 测试集预测 = 5折概率的平均值
    test_proba = np.mean(np.vstack(test_proba_folds), axis=0)
    oof_pred   = (oof_proba >= 0.5).astype(int)
    oof_acc    = accuracy_score(y, oof_pred)

    print(f"\nCV 本地OOF accuracy: {oof_acc:.5f}")
    return y, oof_proba, oof_pred, test_proba


y_true, oof_proba, oof_pred, test_proba = cross_validate_and_predict(
    train_p, test_p, feature_cols, categorical_cols
)

Fold 1: accuracy=0.82001, best_iter=225
Fold 2: accuracy=0.80391, best_iter=288
Fold 3: accuracy=0.81886, best_iter=223
Fold 4: accuracy=0.82048, best_iter=198
Fold 5: accuracy=0.81300, best_iter=252

CV 本地OOF accuracy: 0.81525


# 7. 预测

将测试集概率用阈值0.5转为二分类标签，整理为提交格式。

In [8]:
# 概率>=0.5 => Transported=True，<0.5 => False
final_pred = (test_proba >= 0.5)

submission = pd.DataFrame({
    "PassengerId": test_p["PassengerId"].values,
    "Transported": final_pred,
})

print(f"预测Transported=True的比例: {final_pred.mean():.3f}")
submission.head()

预测Transported=True的比例: 0.515


,PassengerId,Transported
0,0013_01,True
1,0018_01,False
2,0019_01,True
3,0021_01,True
4,0023_01,False


# 8. 提交

保存CSV文件，上传至Kaggle。

> V1.2 CatBoost Kaggle LB = 0.80967（StratifiedKFold，19特征主线基线）

In [9]:
# 保存提交文件
OUTPUT_PATH = "submission_catboost_v1_2.csv"
submission.to_csv(OUTPUT_PATH, index=False)
print(f"已保存: {OUTPUT_PATH}")
print(submission["Transported"].value_counts())

已保存: submission_catboost_v1_2.csv
Transported
True     2204
False    2073
Name: count, dtype: int64
